# Northern Virginia load forecasting — backtest notebook

Study: "Can forecasting models keep up with data-center-driven
demand? Hourly electricity load in the Dominion Energy zone."
Implements PLAN.md exactly (repo: nova-load-forecasting). Data:
pinned PUDL release v2026.11.0 (EIA-930, DOM subregion), hash
verified in-notebook; weather from Open-Meteo (no key). Flagged-hour
rules, training start, test period, MASE scale and scoring follow
DECISIONS.md #4-#8. No credentials of any kind are used or stored.
Outputs are written to the working directory (results files are
pulled back into the repo's results/ folder after the run).

MODE is 'smoke' (a handful of origins, cheap settings, every code
path exercised) or 'full' (the PLAN design). An optional NOVA_CONFIG
environment variable (JSON) may override CONFIG keys for local
testing; the pushed notebook runs with the defaults below.

In [ ]:
import hashlib
import json
import os
import re
import time
import traceback
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CONFIG = {
    "mode": "full",            # 'smoke' or 'full' (set by the build)
    "workdir": "/kaggle/working",
    "pudl_url": ("https://s3.us-west-2.amazonaws.com/pudl.catalyst.coop/"
                 "v2026.11.0/out_eia930__hourly_subregion_demand.parquet"),
    "pudl_sha256": ("1216ed4f47617e3b7271af9f723cd5a59fab025a4eeb98a"
                    "ec145a8e1688957b2"),
    "ba_code": "PJM",
    "subregion_code": "DOM",
    "etz": "America/New_York",
    "train_start": "2019-01-01 00:00",
    "test_start": "2024-11-01 00:00",
    "half2_start": "2025-11-01 00:00",
    "n_origins": 104,
    "horizons": [24, 168],
    "refit_every": 13,         # origins (weeks)
    "weather_points": [        # Ashburn, Richmond, Norfolk (1/3 each)
        (39.04, -77.49), (37.54, -77.44), (36.85, -76.29)],
    "sarimax_candidates": [
        ((1, 0, 1), (1, 1, 1, 24)), ((2, 0, 1), (1, 1, 1, 24)),
        ((1, 0, 2), (1, 1, 1, 24)), ((2, 0, 2), (1, 1, 1, 24)),
        ((1, 1, 1), (1, 1, 1, 24)), ((2, 1, 2), (1, 1, 1, 24)),
        ((1, 0, 1), (0, 1, 1, 24)), ((2, 0, 1), (1, 0, 1, 24))],
    "lstm_seeds": [11, 22, 33],
    "lstm_epochs_full": 12,
    "lstm_epochs_smoke": 2,
    "lstm_stride": 4,
    "lstm_hidden": 64,
    "chronos_model": "amazon/chronos-bolt-base",
    "timesfm_repo_2p5": "google/timesfm-2.5-200m-pytorch",
    "timesfm_repo_2p0": "google/timesfm-2.0-500m-pytorch",
    "llm_model": "Qwen/Qwen2.5-7B-Instruct",
    "llm_samples": 5,
    "bootstrap_B": 2000,
    "bootstrap_block": 4,
    "bootstrap_seed": 20261117,
    # Resume support (defaults off; used only by a resume kernel
    # build after an interrupted run — see DECISIONS #11):
    "resume_checkpoint": None,        # path to a forecasts CSV
    "sarimax_order_override": None,   # (order, seasonal_order)
    "sarimax_min_origin": 0,          # skip SARIMAX origins below this
    "methods": ["snaive", "naive", "sarimax_oracle", "sarimax_nofuture",
                "lstm", "chronos", "timesfm", "llm"],
}
if os.environ.get("NOVA_CONFIG"):
    CONFIG.update(json.loads(os.environ["NOVA_CONFIG"]))

MODE = CONFIG["mode"]
WORKDIR = CONFIG["workdir"]
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

RUN_RECORD = {
    "study": "S04 Northern Virginia load forecasting",
    "mode": MODE,
    "started_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "config_mode": MODE,
    "method_status": {},
    "notes": [],
}
T0 = time.time()


def record(key, value):
    RUN_RECORD[key] = value


def save_run_record():
    RUN_RECORD["elapsed_s"] = round(time.time() - T0, 1)
    with open(os.path.join(WORKDIR, "run_record.json"), "w") as fh:
        json.dump(RUN_RECORD, fh, indent=1, default=str)


print("mode:", MODE, "| workdir:", WORKDIR)

In [ ]:
# Environment: Kaggle images preinstall torch, transformers and
# statsmodels; the two forecasting libraries are installed here.
import subprocess as _sp
import sys as _sys


def _ensure(mod, pkg):
    try:
        __import__(mod)
    except ImportError:
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", pkg],
                check=False)


_ensure("chronos", "chronos-forecasting")
_ensure("timesfm", "timesfm")
_ensure("bitsandbytes", "bitsandbytes")


## 1. Demand data — pinned PUDL release, hash verified

In [ ]:
import requests


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()


parquet_path = os.path.join(WORKDIR, "out_eia930__hourly_subregion_demand.parquet")
if not (os.path.exists(parquet_path)
        and sha256_of(parquet_path) == CONFIG["pudl_sha256"]):
    print("downloading pinned PUDL file ...")
    with requests.get(CONFIG["pudl_url"], stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(parquet_path, "wb") as fh:
            for chunk in r.iter_content(1 << 22):
                fh.write(chunk)
digest = sha256_of(parquet_path)
assert digest == CONFIG["pudl_sha256"], f"SHA-256 mismatch: {digest}"
record("pudl_sha256_verified", digest)
print("SHA-256 verified:", digest)

raw = pd.read_parquet(
    parquet_path,
    columns=["datetime_utc", "balancing_authority_code_eia",
             "balancing_authority_subregion_code_eia",
             "demand_reported_mwh", "demand_imputed_pudl_mwh",
             "demand_imputed_pudl_mwh_imputation_code"],
    filters=[("balancing_authority_code_eia", "==", CONFIG["ba_code"]),
             ("balancing_authority_subregion_code_eia", "==",
              CONFIG["subregion_code"])])
raw = raw.rename(columns={
    "demand_imputed_pudl_mwh_imputation_code": "imputation_code"})
raw = raw.sort_values("datetime_utc").reset_index(drop=True)
print("DOM rows:", len(raw), "| span:", raw["datetime_utc"].min(),
      "->", raw["datetime_utc"].max())

## 2. Weather — Open-Meteo archive, three points, fixed 1/3 weights

In [ ]:
def _fetch_chunk(lat, lon, start, end):
    """One Open-Meteo archive request with retries; on repeated
    failure the range is split in half and fetched recursively."""
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={lat}&longitude={lon}"
           f"&start_date={start}&end_date={end}"
           "&hourly=temperature_2m&timezone=UTC")
    last_err = None
    for attempt in range(4):
        try:
            with requests.get(url, timeout=(20, 120)) as r:
                r.raise_for_status()
                h = r.json()["hourly"]
            return pd.DataFrame({
                "ts_utc": pd.to_datetime(h["time"]),
                "temp": h["temperature_2m"]})
        except Exception as err:  # retry with backoff, then split
            last_err = err
            time.sleep(5 * (attempt + 1))
    d0 = pd.Timestamp(start)
    d1 = pd.Timestamp(end)
    if (d1 - d0).days <= 7:
        raise RuntimeError(f"weather fetch failed {start}..{end}: "
                           f"{last_err}")
    mid = d0 + (d1 - d0) / 2
    mid_s = mid.strftime("%Y-%m-%d")
    left = _fetch_chunk(lat, lon, start, mid_s)
    right = _fetch_chunk(lat, lon,
                         (mid + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                         end)
    return pd.concat([left, right])


def fetch_weather():
    frames = []
    import datetime as dt
    for lat, lon in CONFIG["weather_points"]:
        parts = []
        d = dt.date(2018, 10, 1)
        while d <= dt.date(2026, 10, 31):
            end_d = min(d + dt.timedelta(days=91), dt.date(2026, 10, 31))
            parts.append(_fetch_chunk(lat, lon, d.isoformat(),
                                      end_d.isoformat()))
            print(f"weather {lat},{lon} through {end_d}", flush=True)
            d = end_d + dt.timedelta(days=1)
            time.sleep(0.2)
        s = (pd.concat(parts).drop_duplicates("ts_utc")
               .set_index("ts_utc")["temp"])
        s.index = pd.to_datetime(s.index).tz_localize("UTC")
        frames.append(s)
    w = pd.concat(frames, axis=1)
    return w.mean(axis=1).rename("temp")  # fixed weights 1/3 each


weather_path = os.path.join(WORKDIR, "weather_utc.parquet")
if os.path.exists(weather_path):
    temp_utc = pd.read_parquet(weather_path).iloc[:, 0].rename("temp")
else:
    temp_utc = fetch_weather()
    temp_utc.to_frame().to_parquet(weather_path)
print("weather hours:", len(temp_utc), "| nulls:", int(temp_utc.isna().sum()))

## 3. Series construction — flags, input fill, origins, MASE scale

Timestamps in PUDL are end-of-hour UTC; the series index below is
hour-start in US Eastern. Scoring targets are reported values on
unflagged hours only. Model inputs use reported values with every
flagged/missing hour replaced by the value 168 h earlier, stepping
back another week if that hour is also flagged (DECISIONS #4).

In [ ]:
ETZ = CONFIG["etz"]
ts_utc = (pd.DatetimeIndex(pd.to_datetime(raw["datetime_utc"]))
          .tz_localize("UTC")
          - pd.Timedelta(hours=1))                    # hour start, UTC
index_et = ts_utc.tz_convert(ETZ)
assert index_et.is_monotonic_increasing and not index_et.duplicated().any()

reported = raw["demand_reported_mwh"].to_numpy(dtype=float)
flagged = (raw["imputation_code"].notna().to_numpy()
           | np.isnan(reported))
imputed = raw["demand_imputed_pudl_mwh"].to_numpy(dtype=float)

filled = reported.copy()
n_fallback = 0
for i in np.flatnonzero(flagged):
    j = i - 168
    while j >= 0 and flagged[j]:
        j -= 168
    if j >= 0:
        filled[i] = reported[j]
    else:                       # only possible inside excluded 2018
        filled[i] = imputed[i]
        n_fallback += 1
record("fill_imputed_fallback_hours", n_fallback)

temp = temp_utc.reindex(ts_utc).to_numpy(dtype=float)
assert not np.isnan(
    temp[ts_utc >= pd.Timestamp("2019-01-01", tz="UTC")]).any(), \
    "weather gap"

N = len(reported)
train_start_pos = int(np.searchsorted(
    index_et, pd.Timestamp(CONFIG["train_start"], tz=ETZ)))
test_start_pos = int(np.searchsorted(
    index_et, pd.Timestamp(CONFIG["test_start"], tz=ETZ)))
assert not np.isnan(filled[train_start_pos:]).any()
print(f"series hours: {N} | train start pos {train_start_pos} | "
      f"flagged total {int(flagged.sum())} | fallback fills {n_fallback}")

# Forecast origins: weekly at 00:00 Eastern from the test start; the
# last origin is the latest whose full 168 h horizon has data.
origin_ts = pd.date_range(CONFIG["test_start"], periods=CONFIG["n_origins"],
                          freq="7D", tz=ETZ)
origin_pos = index_et.get_indexer(origin_ts)
assert (origin_pos >= 0).all()
assert origin_pos[-1] + 168 <= N, "last origin horizon exceeds data"
half2_pos = int(np.searchsorted(
    index_et, pd.Timestamp(CONFIG["half2_start"], tz=ETZ)))
origin_half = np.where(origin_pos < half2_pos, 1, 2)
print("origins:", len(origin_pos), "| first:", origin_ts[0],
      "| last:", origin_ts[-1], "| halves:",
      int((origin_half == 1).sum()), "/", int((origin_half == 2).sum()))

# Fixed MASE scale (DECISIONS #7): lag-168 seasonal-naive MAE over the
# training history under the same flagged-hour rules.
_scale_p = np.arange(max(train_start_pos, 168), test_start_pos)
_scale_ok = ~flagged[_scale_p]
MASE_SCALE = float(np.mean(np.abs(
    reported[_scale_p][_scale_ok] - filled[_scale_p - 168][_scale_ok])))
record("mase_scale_mw", MASE_SCALE)
print("MASE scale (MW):", round(MASE_SCALE, 2))

# Dropped target hours per half (distinct hours, 168 h windows).
dropped = {}
for half in (1, 2):
    targets = set()
    for op in origin_pos[origin_half == half]:
        targets.update(range(op, op + 168))
    targets = np.array(sorted(targets))
    dropped[half] = {
        "target_hours": int(len(targets)),
        "dropped_flagged": int(flagged[targets].sum()),
    }
record("dropped_target_hours", dropped)
print("dropped target hours per half:", dropped)

# Working origin subset (smoke mode exercises every code path cheaply)
if MODE == "smoke":
    WORK_ORIGINS = [0, 1, 2, 3, 60]
else:
    WORK_ORIGINS = list(range(len(origin_pos)))
record("work_origins", WORK_ORIGINS)

## 4. Calendar and weather features

In [ ]:
def us_federal_holidays(years):
    """US federal holiday dates (actual calendar dates)."""
    import datetime as dt

    def nth_weekday(y, m, wd, n):
        d = dt.date(y, m, 1)
        return d + dt.timedelta(days=((wd - d.weekday()) % 7) + 7 * (n - 1))

    def last_weekday(y, m, wd):
        d = (dt.date(y, m + 1, 1) - dt.timedelta(days=1)) if m < 12 \
            else dt.date(y, 12, 31)
        return d - dt.timedelta(days=(d.weekday() - wd) % 7)

    out = set()
    for y in years:
        out |= {dt.date(y, 1, 1), dt.date(y, 7, 4), dt.date(y, 11, 11),
                dt.date(y, 12, 25), nth_weekday(y, 1, 0, 3),
                nth_weekday(y, 2, 0, 3), last_weekday(y, 5, 0),
                nth_weekday(y, 9, 0, 1), nth_weekday(y, 10, 0, 2),
                nth_weekday(y, 11, 3, 4)}
        if y >= 2021:
            out.add(dt.date(y, 6, 19))
    return out


HOLIDAYS = us_federal_holidays(range(2018, 2027))
et_dates = index_et.date
HOLIDAY = np.array([d in HOLIDAYS for d in et_dates], dtype=float)
HOD = index_et.hour.to_numpy(dtype=float)
DOW = index_et.dayofweek.to_numpy(dtype=float)
POS = np.arange(N, dtype=float)
DOY_FRAC = (index_et.dayofyear.to_numpy(dtype=float)
            + HOD / 24.0) / 365.25


def degree_hours(t):
    return np.maximum(0.0, 18.3 - t), np.maximum(0.0, t - 18.3)


HDD, CDD = degree_hours(temp)
# No-future-weather stand-in: the temperature 168 h earlier.
temp_lag168 = np.full(N, np.nan)
temp_lag168[168:] = temp[:-168]
HDD_L, CDD_L = degree_hours(np.nan_to_num(temp_lag168, nan=18.3))


def fourier(pos, period, k):
    ang = 2 * np.pi * pos / period
    return [np.sin(k * ang), np.cos(k * ang)]


def exog_rows(positions, temp_values):
    """SARIMAX exogenous rows (fixed column order) for the given
    absolute positions and temperature values."""
    hdd, cdd = degree_hours(temp_values)
    cols = []
    for k in (1, 2, 3):
        cols += fourier(POS[positions], 168.0, k)
    for k in (1, 2):
        ang = 2 * np.pi * DOY_FRAC[positions]
        cols += [np.sin(k * ang), np.cos(k * ang)]
    cols += [hdd, cdd, HOLIDAY[positions]]
    return np.column_stack(cols)


# History exog always uses observed temperature (it is known history
# for both variants); the variants differ only in horizon temperature.
EXOG_HIST = exog_rows(np.arange(N), temp)


def horizon_exog(target_positions, variant):
    if variant == "oracle":
        t = temp[target_positions]
    else:  # nofuture: last week's temperature as the stand-in
        t = temp_lag168[target_positions]
    return exog_rows(target_positions, t)


CTX = 168  # context length shared by LSTM / Chronos / TimesFM / LLM


def calendar_feats(positions):
    """(hod sin/cos, dow sin/cos, holiday) for absolute positions."""
    hod = HOD[positions]
    dow = DOW[positions]
    return np.column_stack([
        np.sin(2 * np.pi * hod / 24), np.cos(2 * np.pi * hod / 24),
        np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
        HOLIDAY[positions]])

## 5. Forecast store and baselines

Every method writes into one long table: one row per
(method, origin, step). Point forecasts come from the filled input
series only; targets and flags are attached at scoring time.

In [ ]:
FC_ROWS = []
RESUMED = {}  # method -> set of origin_idx loaded from a checkpoint

if CONFIG.get("resume_checkpoint"):
    _path = CONFIG["resume_checkpoint"]
    if not os.path.exists(_path):
        import glob
        _hits = glob.glob("/kaggle/input/**/forecasts_checkpoint.csv",
                          recursive=True)
        assert _hits, f"resume checkpoint not found at {_path}"
        _path = _hits[0]
    _ck = pd.read_csv(_path)
    _ck["origin"] = pd.to_datetime(_ck["origin"], utc=True).dt.tz_convert(ETZ)
    # target_pos is a deterministic function of origin and step;
    # recompute it rather than trusting the stored column.
    _ck["target_pos"] = [int(origin_pos[int(oi)]) + int(st)
                         for oi, st in zip(_ck["origin_idx"],
                                           _ck["step"])]
    for (_m, _oi), _g in _ck.groupby(["method", "origin_idx"]):
        FC_ROWS.append(_g.reset_index(drop=True))
        RESUMED.setdefault(_m, set()).add(int(_oi))
    record("resumed_origins", {m: sorted(v) for m, v in RESUMED.items()})
    print("resumed checkpoint origins:",
          {m: len(v) for m, v in RESUMED.items()})


def add_forecasts(method, oi, point, lo=None, hi=None):
    """point/lo/hi: arrays of length 168 for origin index oi."""
    op = int(origin_pos[oi])
    steps = np.arange(1, 169)
    FC_ROWS.append(pd.DataFrame({
        "method": method,
        "origin_idx": oi,
        "origin": origin_ts[oi],
        "half": int(origin_half[oi]),
        "step": steps,
        "target_pos": op + steps - 1,
        "point": np.asarray(point, dtype=float),
        "lo": np.nan if lo is None else np.asarray(lo, dtype=float),
        "hi": np.nan if hi is None else np.asarray(hi, dtype=float),
    }))


def checkpoint():
    """Persist forecasts so far; if the session dies, the kernel
    output still holds every completed method's forecasts."""
    if FC_ROWS:
        pd.concat(FC_ROWS, ignore_index=True).to_csv(
            os.path.join(WORKDIR, "forecasts_checkpoint.csv"),
            index=False)
    save_run_record()


def run_method(name, fn):
    t = time.time()
    try:
        fn()
        RUN_RECORD["method_status"][name] = "ok"
    except Exception:
        RUN_RECORD["method_status"][name] = "failed"
        RUN_RECORD["notes"].append(
            f"{name} failed:\n{traceback.format_exc()}")
        print(f"!!! {name} FAILED\n{traceback.format_exc()}")
    RUN_RECORD.setdefault("method_seconds", {})[name] = round(
        time.time() - t, 1)
    checkpoint()


def residual_quantiles(resid, upto_pos):
    r = resid[train_start_pos:upto_pos]
    r = r[~np.isnan(r)]
    return float(np.quantile(r, 0.10)), float(np.quantile(r, 0.90))


def do_baselines():
    resid_s = np.full(N, np.nan)
    ok = ~flagged
    resid_s[168:] = np.where(ok[168:], reported[168:] - filled[:-168],
                             np.nan)
    resid_n = np.full(N, np.nan)
    resid_n[24:] = np.where(ok[24:], reported[24:] - filled[:-24],
                            np.nan)
    for oi in WORK_ORIGINS:
        if oi in RESUMED.get("snaive", set()):
            continue
        op = int(origin_pos[oi])
        tp = op + np.arange(168)
        q10, q90 = residual_quantiles(resid_s, op)
        point = filled[tp - 168]
        add_forecasts("snaive", oi, point, point + q10, point + q90)
        q10n, q90n = residual_quantiles(resid_n, op)
        point_n = filled[tp - 24]
        lo_n, hi_n = point_n + q10n, point_n + q90n
        # Naive is a 24 h baseline only (PLAN): mask later steps so
        # they can never be scored (they would use future data).
        point_n[24:], lo_n[24:], hi_n[24:] = np.nan, np.nan, np.nan
        add_forecasts("naive", oi, point_n, lo_n, hi_n)


if "snaive" in CONFIG["methods"]:
    run_method("baselines", do_baselines)

## 6. SARIMAX — order by AIC on training data only, refit 13-weekly

Daily seasonality (s = 24), Fourier terms for weekly and annual
cycles, heating/cooling degree-hours, holiday dummies. The two
variants share each refit's fitted parameters and differ only in
the temperature used over the horizon: observed (oracle, an upper
bound) vs the value 168 h earlier (no future weather). Between
refits the fitted model is extended with new data (no refit) up to
each origin.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX


def sarimax_order_selection():
    import gc
    lo = int(np.searchsorted(index_et,
                             pd.Timestamp("2022-11-01 00:00", tz=ETZ)))
    y = filled[lo:test_start_pos]
    X = EXOG_HIST[lo:test_start_pos]
    best = None
    for order, sorder in CONFIG["sarimax_candidates"]:
        try:
            res = SARIMAX(y, exog=X, order=order, seasonal_order=sorder,
                          enforce_stationarity=False,
                          enforce_invertibility=False,
                          concentrate_scale=True).fit(disp=False)
            print("candidate", order, sorder, "AIC", round(res.aic, 1))
            if best is None or res.aic < best[0]:
                best = (res.aic, order, sorder, res.params)
            del res
            gc.collect()
        except Exception:
            print("candidate failed", order, sorder)
            traceback.print_exc()
    record("sarimax_order", {"aic": best[0], "order": best[1],
                             "seasonal_order": best[2]})
    print("selected:", best[:3])
    return best[1], best[2], best[3]


def do_sarimax():
    # Optimizer numerics only (same model, data, orders and refit
    # schedule): sigma2 concentrated out, and each fit warm-started
    # from the previous fit's parameters (already-transformed space,
    # statsmodels' start_params convention).
    record("sarimax_estimation",
           {"concentrate_scale": True, "warm_start": True,
            "maxiter": "statsmodels default (50)"})
    if CONFIG.get("sarimax_order_override"):
        order, sorder = CONFIG["sarimax_order_override"]
        prev_params = None
        record("sarimax_order", {"order": order,
                                 "seasonal_order": sorder,
                                 "note": "resume: order taken from the "
                                         "interrupted run's selection"})
    elif MODE == "smoke":
        order, sorder = (1, 0, 1), (1, 1, 1, 24)
        prev_params = None
        record("sarimax_order", {"order": order, "seasonal_order": sorder,
                                 "note": "smoke: selection skipped"})
    else:
        order, sorder, prev_params = sarimax_order_selection()
    every = CONFIG["refit_every"]
    min_oi = CONFIG.get("sarimax_min_origin", 0)
    refit_idx = sorted({(oi // every) * every for oi in WORK_ORIGINS
                        if oi >= min_oi})
    import gc
    for ri in refit_idx:
        rpos = int(origin_pos[ri])
        mod = SARIMAX(filled[train_start_pos:rpos],
                      exog=EXOG_HIST[train_start_pos:rpos],
                      order=order, seasonal_order=sorder,
                      enforce_stationarity=False,
                      enforce_invertibility=False,
                      concentrate_scale=True)
        if prev_params is not None and len(prev_params) == mod.k_params:
            res = mod.fit(disp=False, start_params=prev_params)
        else:
            res = mod.fit(disp=False)
        prev_params = res.params
        block = [oi for oi in WORK_ORIGINS
                 if ri <= oi < ri + every and oi >= min_oi]
        for oi in block:
            op = int(origin_pos[oi])
            if op > rpos:
                applied = res.apply(
                    filled[rpos:op], exog=EXOG_HIST[rpos:op],
                    refit=False)
            else:
                applied = res
            tp = op + np.arange(168)
            for variant, mname in (("oracle", "sarimax_oracle"),
                                   ("nofuture", "sarimax_nofuture")):
                if mname not in CONFIG["methods"]:
                    continue
                fc = applied.get_forecast(
                    steps=168, exog=horizon_exog(tp, variant))
                sf = fc.summary_frame(alpha=0.20)
                add_forecasts(mname, oi, sf["mean"].to_numpy(),
                              sf["mean_ci_lower"].to_numpy(),
                              sf["mean_ci_upper"].to_numpy())
            if applied is not res:
                del applied
            gc.collect()
        del res
        gc.collect()
        checkpoint()
        print(f"sarimax refit at origin {ri} done", flush=True)


if any(m in CONFIG["methods"]
       for m in ("sarimax_oracle", "sarimax_nofuture")):
    run_method("sarimax", do_sarimax)

## 7. LSTM — small, quantile loss (10/50/90), 3 seeds, refit 13-weekly

Inputs are the past 168 h: load, temperature (raw, heating and
cooling degree-hours — the temperature features as in SARIMAX),
calendar features. The output head also sees the known future
calendar features. No future temperature is used, matching the
no-future-weather SARIMAX variant. Point forecast: mean over seeds
of the median; interval: mean over seeds of the 10/90 quantiles.

In [ ]:
def do_lstm():
    import torch
    from torch import nn

    device = "cuda" if torch.cuda.is_available() else "cpu"
    record("lstm_device", device)
    epochs = (CONFIG["lstm_epochs_smoke"] if MODE == "smoke"
              else CONFIG["lstm_epochs_full"])
    stride = 24 if MODE == "smoke" else CONFIG["lstm_stride"]
    hidden = CONFIG["lstm_hidden"]

    def build_features(load_scale, temp_mean, temp_std):
        load_s = filled / load_scale
        temp_s = (temp - temp_mean) / temp_std
        feats = np.column_stack([
            load_s, temp_s, HDD / 10.0, CDD / 10.0,
            np.sin(2 * np.pi * HOD / 24), np.cos(2 * np.pi * HOD / 24),
            np.sin(2 * np.pi * DOW / 7), np.cos(2 * np.pi * DOW / 7),
            HOLIDAY]).astype(np.float32)
        return feats

    class Forecaster(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = nn.LSTM(9, hidden, num_layers=1, batch_first=True)
            self.head = nn.Sequential(
                nn.Linear(hidden + 5, 64), nn.ReLU(), nn.Linear(64, 3))

        def forward(self, x, fut):
            _, (h, _) = self.enc(x)
            h = h[-1].unsqueeze(1).expand(-1, CTX, -1)
            return self.head(torch.cat([h, fut], dim=-1))

    def pinball(pred, target, qs=(0.1, 0.5, 0.9)):
        loss = 0.0
        for i, q in enumerate(qs):
            err = target - pred[..., i]
            loss = loss + torch.maximum(q * err, (q - 1) * err).mean()
        return loss

    spread_rows = []
    every = CONFIG["refit_every"]
    refit_idx = sorted({(oi // every) * every for oi in WORK_ORIGINS})
    for ri in refit_idx:
        rpos = int(origin_pos[ri])
        train_slice = slice(train_start_pos, rpos)
        load_scale = float(np.mean(filled[train_slice]))
        temp_mean = float(np.mean(temp[train_slice]))
        temp_std = float(np.std(temp[train_slice]))
        feats = build_features(load_scale, temp_mean, temp_std)
        fut_all = calendar_feats(np.arange(N)).astype(np.float32)
        ends = np.arange(train_start_pos + CTX - 1, rpos - CTX, stride)
        X = np.stack([feats[e - CTX + 1:e + 1] for e in ends])
        F = np.stack([fut_all[e + 1:e + 1 + CTX] for e in ends])
        Y = np.stack([filled[e + 1:e + 1 + CTX] for e in ends])
        Y = (Y / load_scale).astype(np.float32)
        print(f"lstm refit {ri}: {len(ends)} training windows", flush=True)
        block = [oi for oi in WORK_ORIGINS
                 if ri <= oi < ri + every]
        preds = {oi: [] for oi in block}
        for seed in CONFIG["lstm_seeds"]:
            torch.manual_seed(seed)
            np.random.seed(seed)
            model = Forecaster().to(device)
            opt = torch.optim.Adam(model.parameters(), lr=1e-3)
            Xt = torch.tensor(X, device=device)
            Ft = torch.tensor(F, device=device)
            Yt = torch.tensor(Y, device=device)
            n = len(ends)
            for ep in range(epochs):
                perm = torch.randperm(n, device=device)
                for b in range(0, n, 256):
                    idx = perm[b:b + 256]
                    opt.zero_grad()
                    out = model(Xt[idx], Ft[idx])
                    loss = pinball(out, Yt[idx])
                    loss.backward()
                    opt.step()
            model.eval()
            with torch.no_grad():
                for oi in block:
                    op = int(origin_pos[oi])
                    x = torch.tensor(feats[op - CTX:op][None],
                                     device=device)
                    f = torch.tensor(fut_all[op:op + CTX][None],
                                     device=device)
                    out = model(x, f)[0].cpu().numpy() * load_scale
                    preds[oi].append(np.sort(out, axis=-1))
            del model, Xt, Ft, Yt
            if device == "cuda":
                torch.cuda.empty_cache()
        for oi in block:
            arr = np.stack(preds[oi])            # seeds x 168 x 3
            point = arr[:, :, 1].mean(axis=0)
            lo = arr[:, :, 0].mean(axis=0)
            hi = arr[:, :, 2].mean(axis=0)
            add_forecasts("lstm", oi, point, lo, hi)
            spread_rows.append({
                "origin_idx": oi,
                "mean_abs_seed_spread_mw": float(
                    arr[:, :, 1].std(axis=0).mean())})
        checkpoint()
        print(f"lstm refit {ri} done", flush=True)
    pd.DataFrame(spread_rows).to_csv(
        os.path.join(WORKDIR, "lstm_seed_spread.csv"), index=False)


if "lstm" in CONFIG["methods"]:
    run_method("lstm", do_lstm)

## 8. Chronos-Bolt — zero-shot, same 168 h context as the LLM

In [ ]:
def do_chronos():
    import torch
    from chronos import BaseChronosPipeline

    pipe = BaseChronosPipeline.from_pretrained(
        CONFIG["chronos_model"], device_map="cuda",
        torch_dtype=torch.bfloat16)
    record("chronos_model", CONFIG["chronos_model"])
    record("chronos_context", CTX)
    contexts = np.stack([filled[int(origin_pos[oi]) - CTX:
                                int(origin_pos[oi])]
                         for oi in WORK_ORIGINS]).astype(np.float32)
    quantiles, _mean = pipe.predict_quantiles(
        torch.tensor(contexts), prediction_length=168,
        quantile_levels=[0.1, 0.5, 0.9])
    q = quantiles.cpu().numpy()                # origins x 168 x 3
    for k, oi in enumerate(WORK_ORIGINS):
        add_forecasts("chronos", oi, q[k, :, 1], q[k, :, 0], q[k, :, 2])


if "chronos" in CONFIG["methods"]:
    run_method("chronos", do_chronos)

## 9. TimesFM — zero-shot, same 168 h context

In [ ]:
def do_timesfm():
    import timesfm

    record("timesfm_version", getattr(timesfm, "__version__", "unknown"))
    contexts = [filled[int(origin_pos[oi]) - CTX:int(origin_pos[oi])]
                .astype(np.float64) for oi in WORK_ORIGINS]
    try:
        model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
            CONFIG["timesfm_repo_2p5"])
        model.compile(timesfm.ForecastConfig(
            max_context=512, max_horizon=168, normalize_inputs=True,
            use_continuous_quantile_head=True,
            fix_quantile_crossing=True))
        _point, q = model.forecast(horizon=168, inputs=contexts)
        record("timesfm_model", CONFIG["timesfm_repo_2p5"])
    except Exception:
        print("TimesFM 2.5 path failed; trying 2.0 API")
        traceback.print_exc()
        tfm = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend="gpu", per_core_batch_size=32, horizon_len=168,
                context_len=512, num_layers=50, model_dims=1280),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id=CONFIG["timesfm_repo_2p0"]))
        _point, q = tfm.forecast(contexts, freq=[0] * len(contexts))
        record("timesfm_model", CONFIG["timesfm_repo_2p0"])
    record("timesfm_context", CTX)
    q = np.asarray(q)                          # origins x 168 x 10
    # columns: [mean, q0.1, ..., q0.9]
    for k, oi in enumerate(WORK_ORIGINS):
        add_forecasts("timesfm", oi, q[k, :, 5], q[k, :, 1], q[k, :, 9])


if "timesfm" in CONFIG["methods"]:
    run_method("timesfm", do_timesfm)

## 10. General LLM — reduced test (26 origins, 24 h only)

Past 168 h as scaled text (LLMTime-style), 5 samples per origin,
interval = 10th–90th percentile of the samples. Local open model
in 4-bit on the GPU. No API credentials are used anywhere in this
notebook; if the local model cannot produce parseable forecasts
the arm is recorded as incomplete.

In [ ]:
LLM_PROMPT = (
    "Hourly electricity demand for a region, in hundreds of MW, "
    "for the past 168 hours:\n{series}\n"
    "Continue the series with the next 24 hourly values, in the "
    "same units and the same format: comma-separated numbers only, "
    "no other text.")


def do_llm():
    import torch
    from transformers import (AutoModelForCausalLM, AutoTokenizer,
                              BitsAndBytesConfig)

    model_id = CONFIG["llm_model"]
    n_samples = 2 if MODE == "smoke" else CONFIG["llm_samples"]
    llm_origins = [oi for oi in WORK_ORIGINS if oi % 4 == 0]
    if MODE == "smoke":
        llm_origins = llm_origins[:2]
    tok = AutoTokenizer.from_pretrained(model_id)
    bnb = BitsAndBytesConfig(load_in_4bit=True,
                             bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map="cuda")
    model.eval()
    decoding = {"do_sample": True, "temperature": 0.8, "top_p": 0.9,
                "max_new_tokens": 160, "num_return_sequences": n_samples}
    record("llm", {"model": model_id, "quantization": "4-bit NF4",
                   "prompt_template": LLM_PROMPT,
                   "decoding": {k: v for k, v in decoding.items()},
                   "origins": llm_origins, "scale": "value/100, rounded"})
    n_unparseable = 0
    n_total = 0
    for oi in llm_origins:
        op = int(origin_pos[oi])
        series = ",".join(str(int(round(v / 100.0)))
                          for v in filled[op - CTX:op])
        messages = [
            {"role": "system",
             "content": "You are a time-series forecasting assistant."},
            {"role": "user", "content": LLM_PROMPT.format(series=series)}]
        prompt = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, **decoding,
                                 pad_token_id=tok.eos_token_id)
        samples = []
        for row in out:
            text = tok.decode(row[inputs["input_ids"].shape[1]:],
                              skip_special_tokens=True)
            nums = [float(x) for x in
                    re.findall(r"-?\d+(?:\.\d+)?", text)]
            n_total += 1
            if len(nums) >= 24:
                samples.append(np.array(nums[:24]) * 100.0)
            else:
                n_unparseable += 1
        point = np.full(168, np.nan)
        lo = np.full(168, np.nan)
        hi = np.full(168, np.nan)
        if samples:
            S = np.stack(samples)
            point[:24] = np.median(S, axis=0)
            lo[:24] = np.percentile(S, 10, axis=0)
            hi[:24] = np.percentile(S, 90, axis=0)
        add_forecasts("llm", oi, point, lo, hi)
        print(f"llm origin {oi}: {len(samples)}/{n_samples} parseable",
              flush=True)
    share = (n_unparseable / n_total) if n_total else 1.0
    RUN_RECORD["llm"]["unparseable_share"] = round(share, 4)
    if share > 0.5:
        RUN_RECORD["notes"].append(
            "LLM arm: majority of samples unparseable; arm incomplete.")
    save_run_record()


if "llm" in CONFIG["methods"]:
    run_method("llm", do_llm)

## 11. Scoring — fixed MASE scale, paired block bootstrap

Scores use only unflagged reported targets (DECISIONS #4). Paired
differences are per-origin MASE differences against seasonal naive
with 95% block-bootstrap intervals (blocks of 4 origins).

In [ ]:
METHOD_LABELS = {
    "snaive": "Seasonal naive (lag 168)",
    "naive": "Naive (lag 24)",
    "sarimax_oracle": "SARIMAX (oracle weather)",
    "sarimax_nofuture": "SARIMAX (no future weather)",
    "lstm": "LSTM (3 seeds)",
    "chronos": "Chronos-Bolt base (zero-shot)",
    "timesfm": "TimesFM (zero-shot)",
    "llm": "LLM Qwen2.5-7B 4-bit (reduced test)",
}
INPUT_HISTORY = {
    "snaive": "168 h context",
    "naive": "168 h context",
    "sarimax_oracle": "expanding from 2019-01-01, refit 13-weekly",
    "sarimax_nofuture": "expanding from 2019-01-01, refit 13-weekly",
    "lstm": "expanding from 2019-01-01, retrain 13-weekly",
    "chronos": "168 h context, zero-shot",
    "timesfm": "168 h context, zero-shot",
    "llm": "168 h context, zero-shot, 26 origins",
}

fc = pd.concat(FC_ROWS, ignore_index=True)
fc["y"] = reported[fc["target_pos"].to_numpy()]
fc["flagged"] = flagged[fc["target_pos"].to_numpy()]
fc["scored"] = (~fc["flagged"]) & fc["y"].notna() & fc["point"].notna()
fc["err"] = fc["point"] - fc["y"]
fc["target_time"] = index_et[fc["target_pos"].to_numpy()]
fc.to_csv(os.path.join(WORKDIR, "forecasts_long.csv"), index=False)
print("forecast rows:", len(fc), "| scored:", int(fc["scored"].sum()))


def winkler(y, lo, hi, alpha=0.20):
    width = hi - lo
    return (width
            + np.where(y < lo, (2 / alpha) * (lo - y), 0.0)
            + np.where(y > hi, (2 / alpha) * (y - hi), 0.0))


def block_bootstrap_ci(diffs):
    diffs = np.asarray(diffs, dtype=float)
    n = len(diffs)
    if n < 4:
        return (np.nan, np.nan)
    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    block = CONFIG["bootstrap_block"]
    B = CONFIG["bootstrap_B"]
    nblocks = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(B, nblocks))
    idx = (starts[:, :, None] + np.arange(block)[None, None, :])
    idx = (idx.reshape(B, -1)[:, :n]) % n
    means = diffs[idx].mean(axis=1)
    return (float(np.percentile(means, 2.5)),
            float(np.percentile(means, 97.5)))


def per_origin_mae(df):
    return df.groupby("origin_idx")["err"].apply(
        lambda s: float(np.mean(np.abs(s))))


METHOD_HORIZONS = {"naive": (24,), "llm": (24,)}  # PLAN: 24 h only

rows, paired_rows = [], []
snaive_mae = {}
for horizon in CONFIG["horizons"]:
    sub_h = fc[fc["step"] <= horizon]
    base = sub_h[(sub_h["method"] == "snaive") & sub_h["scored"]]
    for half in (1, 2):
        snaive_mae[(horizon, half)] = per_origin_mae(
            base[base["half"] == half])
for (method, horizon, half), _ in sorted(
        {(m, h, hf): None for m in fc["method"].unique()
         for h in CONFIG["horizons"] for hf in (1, 2)}.items()):
    if horizon not in METHOD_HORIZONS.get(method, CONFIG["horizons"]):
        continue
    sub = fc[(fc["method"] == method) & (fc["step"] <= horizon)
             & (fc["half"] == half) & fc["scored"]]
    if sub.empty:
        continue
    mae_o = per_origin_mae(sub)
    mase_o = mae_o / MASE_SCALE
    lo_b, hi_b = block_bootstrap_ci(mase_o.to_numpy())
    has_int = sub["lo"].notna().any()
    cov = np.mean((sub["y"] >= sub["lo"]) & (sub["y"] <= sub["hi"])) \
        if has_int else np.nan
    wink = np.mean(winkler(sub["y"].to_numpy(), sub["lo"].to_numpy(),
                           sub["hi"].to_numpy())) if has_int else np.nan
    # peak-hour error: per origin and target date, error at the
    # highest-demand scored hour
    sub2 = sub.assign(date=sub["target_time"].dt.date)
    peak_errs = []
    for (_oi, _d), g in sub2.groupby(["origin_idx", "date"]):
        peak_errs.append(abs(g.loc[g["y"].idxmax(), "err"]))
    rows.append({
        "method": method, "method_label": METHOD_LABELS[method],
        "horizon_h": horizon, "half": half,
        "mase": float(mase_o.mean()), "mase_ci_lo": lo_b,
        "mase_ci_hi": hi_b,
        "mape_pct": float(np.mean(np.abs(sub["err"]) / sub["y"]) * 100),
        "peak_hour_err_mw": float(np.mean(peak_errs)),
        "coverage80": float(cov) if has_int else np.nan,
        "winkler80": float(wink) if has_int else np.nan,
        "n_origins": int(sub["origin_idx"].nunique()),
        "input_history": INPUT_HISTORY[method],
        "compute_s": RUN_RECORD.get("method_seconds", {}).get(
            {"sarimax_oracle": "sarimax", "sarimax_nofuture": "sarimax",
             "snaive": "baselines", "naive": "baselines"}.get(
                 method, method), np.nan),
    })
    if method != "snaive":
        # snaive_mae holds per-origin MAE in MW; convert to MASE
        # units before differencing (the two were mixed in the
        # first full run's paired table — DECISIONS #14).
        base_o = snaive_mae[(horizon, half)] / MASE_SCALE
        common = mase_o.index.intersection(base_o.index)
        if len(common) >= 4:
            d = (mase_o.loc[common] - base_o.loc[common]).to_numpy()
            dlo, dhi = block_bootstrap_ci(d)
            paired_rows.append({
                "method": method, "horizon_h": horizon, "half": half,
                "mase_diff_vs_snaive": float(d.mean()),
                "diff_ci_lo": dlo, "diff_ci_hi": dhi,
                "n_origins": int(len(common))})

scores = pd.DataFrame(rows)
paired = pd.DataFrame(paired_rows)
scores.to_csv(os.path.join(WORKDIR, "scores_main.csv"), index=False)
paired.to_csv(os.path.join(WORKDIR, "scores_paired.csv"), index=False)
pd.DataFrame([{"half": h, **dropped[h]} for h in (1, 2)]).to_csv(
    os.path.join(WORKDIR, "dropped_hours.csv"), index=False)
print(scores.to_string())

## 12. Charts and run record

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True)
methods_order = [m for m in METHOD_LABELS if m in set(scores["method"])]
for ax, horizon in zip(axes, CONFIG["horizons"]):
    sub = scores[scores["horizon_h"] == horizon]
    ypos = np.arange(len(methods_order))
    for half, marker, dx in ((1, "o", -0.18), (2, "s", 0.18)):
        s2 = sub[sub["half"] == half].set_index("method")
        xs, los, his, ys = [], [], [], []
        for i, m in enumerate(methods_order):
            if m in s2.index:
                r = s2.loc[m]
                xs.append(r["mase"]); los.append(r["mase_ci_lo"])
                his.append(r["mase_ci_hi"]); ys.append(i + dx)
        if xs:
            ax.errorbar(xs, ys, xerr=[np.array(xs) - np.array(los),
                                      np.array(his) - np.array(xs)],
                        fmt=marker, capsize=3,
                        label=f"half {half}")
    ax.set_yticks(ypos)
    ax.set_yticklabels([METHOD_LABELS[m] for m in methods_order],
                       fontsize=8)
    ax.axvline(1.0, color="grey", lw=0.8, ls="--")
    ax.set_xlabel("MASE (fixed scale; 1.0 = seasonal naive)")
    ax.set_title(f"{horizon} h horizon")
    ax.invert_yaxis()
axes[0].legend(loc="lower right", fontsize=8)
fig.suptitle("MASE by method, split by test half (95% block-bootstrap CI)")
fig.tight_layout()
fig.savefig(os.path.join(WORKDIR, "chart_mase.png"), dpi=150)
plt.close(fig)

cov = scores.dropna(subset=["coverage80"])
fig, ax = plt.subplots(figsize=(10, 4.5))
labels, vals, cols = [], [], []
for _, r in cov.sort_values(["method", "horizon_h", "half"]).iterrows():
    labels.append(f"{r['method']}\n{r['horizon_h']}h H{int(r['half'])}")
    vals.append(r["coverage80"])
    cols.append("tab:blue" if r["half"] == 1 else "tab:orange")
ax.bar(range(len(vals)), vals, color=cols)
ax.axhline(0.80, color="red", ls="--", lw=1, label="80% target")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("empirical coverage of 80% intervals")
ax.set_ylim(0, 1.0)
ax.legend()
ax.set_title("Interval coverage by method, horizon and half")
fig.tight_layout()
fig.savefig(os.path.join(WORKDIR, "chart_coverage.png"), dpi=150)
plt.close(fig)

try:
    import torch
    record("accelerator", torch.cuda.get_device_name(0)
           if torch.cuda.is_available() else "CPU only")
except Exception:
    record("accelerator", "unknown (torch unavailable)")
record("finished_utc", pd.Timestamp.now(tz="UTC").isoformat())
record("forecast_rows", int(len(fc)))
save_run_record()
print("done. elapsed s:", RUN_RECORD["elapsed_s"])